In [ ]:
import numpy as np 
import pandas as pd 

: 

In [ ]:
import numpy as np
import pandas as pd
from mofapy2.run.entry_point import entry_point

np.random.seed(43)
# Barcodes
true_barcodes = (
    open(
        "/vast/projects/Sisseq/human-haematopoiesis-sis-seq/data/preprocessed/barcodes.csv"
    )
    .read()
    .strip()
    .split(",")[2:]
)

# Timepoints
timepoints = (
    open(
        "/vast/projects/Sisseq/human-haematopoiesis-sis-seq/data/preprocessed/timepoints.csv"
    )
    .read()
    .strip()
    .split(",")[2:]
)

# Lineages
lineages = (
    open(
        "/vast/projects/Sisseq/human-haematopoiesis-sis-seq/data/preprocessed/celltypes.csv"
    )
    .read()
    .strip()
    .split(",")[2:]
)


x_bc_3d = np.load(
    "/vast/projects/Sisseq/human-haematopoiesis-sis-seq/data/preprocessed/X_bc.npy"
)

# Flatten
x_bc_2d = x_bc_3d.reshape(x_bc_3d.shape[0], -1)

# Fate feature names
fate_features = [
    f"{t}_{l}"
    for t in timepoints
    for l in lineages
]

# Fate dataframe
df_fate = pd.DataFrame(
    np.log2(x_bc_2d + 1),
    index=true_barcodes,
    columns=fate_features
)


df_rna = pd.read_csv(
    "/vast/projects/Sisseq/human-haematopoiesis-sis-seq/data/preprocessed/X_rna.csv",
    index_col=0
)

df_adt = pd.read_csv(
    "/vast/projects/Sisseq/human-haematopoiesis-sis-seq/data/preprocessed/X_adt.csv",
    index_col=0
)

# Force matching barcodes
df_rna.index = true_barcodes
df_adt.index = true_barcodes

# getting rid of zero var cells - 
df_rna = df_rna.loc[:, df_rna.var() > 0]
df_adt = df_adt.loc[:, df_adt.var() > 0]
df_fate = df_fate.loc[:, df_fate.var() > 0]

print(f"Remaining RNA features: {df_rna.shape[1]}")
print(f"Remaining ADT features: {df_adt.shape[1]}")
print(f"Remaining Fate features: {df_fate.shape[1]}")


feat_rna = df_rna.columns.tolist()
feat_adt = df_adt.columns.tolist()
feat_fate = df_fate.columns.tolist()


mat_rna = df_rna.values.astype(np.float64)
mat_adt = df_adt.values.astype(np.float64)
mat_fate = df_fate.values.astype(np.float64)

# Nested structure:
# views -> groups
data_nested_matrix = [
    [mat_rna],
    [mat_adt],
    [mat_fate]
]
assert list(df_rna.index) == list(df_adt.index)
assert list(df_rna.index) == list(df_fate.index)

# Run MOFA

ent = entry_point()

ent.set_data_options(
    scale_views=False
)

ent.set_data_matrix(
    data_nested_matrix,
    likelihoods=["gaussian", "gaussian", "gaussian"],
    views_names=["RNA", "ADT", "Fate_Tracking"],
    groups_names=["single_group"],
    samples_names=[true_barcodes],
    features_names=[feat_rna, feat_adt, feat_fate]
)

ent.set_model_options(
    factors=15,
    spikeslab_weights=True
)

ent.set_train_options(
    convergence_mode="medium",
    iter=1000,
    verbose=True,
    seed=43
)

ent.build()
ent.run()

ent.save(
    "recent_var_mofa.hdf5",
    save_data=True
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans

# ==========================================
# MOFA latent factors
# ==========================================

Z = ent.model.getExpectations()["Z"]["E"]

print("Z shape:", Z.shape)
# expected: (n_cells, n_factors)

# ==========================================
# Cluster cells in latent space
# ==========================================

n_clusters = 6

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=0,
    n_init="auto"
)

clusts = kmeans.fit_predict(Z)

# ==========================================
# Original lineage tensor
# ==========================================

# shape should be:
# (cells, timepoints, lineages)

X_lineage = x_bc_3d

print("Lineage tensor:", X_lineage.shape)

# ==========================================
# Plot cluster-average fate maps
# ==========================================

ncols = 3
nrows = int(np.ceil(n_clusters / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(5*ncols, 4*nrows)
)

axes = axes.flatten()

for cluster_idx in range(n_clusters):

    ax = axes[cluster_idx]

    # select cells in cluster
    mask = clusts == cluster_idx

    print(f"Cluster {cluster_idx}: {mask.sum()} cells")

    # average fate trajectories
    avg = X_lineage[mask].mean(axis=0)

    # avg shape:
    # (timepoints, lineages)

    sns.heatmap(
        avg.T,
        cmap="viridis",
        ax=ax,
        cbar=(cluster_idx == 0),
        xticklabels=timepoints,
        yticklabels=lineages
    )

    ax.set_title(f"Cluster {cluster_idx}")
    ax.set_xlabel("Timepoint")
    ax.set_ylabel("Lineage")

# remove empty axes
for i in range(n_clusters, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ==========================================
# Flatten lineage tensor
# ==========================================

# shape:
# (cells, timepoints * lineages)

X_fate_flat = x_bc_3d.reshape(
    x_bc_3d.shape[0],
    -1
)

# feature names
fate_features = [
    f"{t}_{l}"
    for t in timepoints
    for l in lineages
]

# ==========================================
# Compute correlations
# ==========================================

n_factors = Z.shape[1]

corr_mat = np.zeros(
    (n_factors, X_fate_flat.shape[1])
)

for k in range(n_factors):

    for j in range(X_fate_flat.shape[1]):

        corr_mat[k, j] = np.corrcoef(
            Z[:, k],
            X_fate_flat[:, j]
        )[0, 1]

# convert to dataframe
corr_df = pd.DataFrame(
    corr_mat,
    index=[f"Factor {i}" for i in range(n_factors)],
    columns=fate_features
)

# ==========================================
# Plot
# ==========================================

plt.figure(figsize=(18, 6))

sns.heatmap(
    corr_df,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1
)

plt.title("MOFA Factor to Fate Correlations")
plt.xlabel("Fate Features")
plt.ylabel("MOFA Factors")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd


def corr_with_view(z, X, feature_names):
    """
    Compute correlation between factor z and each feature in X.
    """
    vals = []
    for j in range(X.shape[1]):
        vals.append(np.corrcoef(z, X[:, j])[0, 1])

    return pd.Series(vals, index=feature_names)


def top_n(series, n=20):
    """
    Return top n features ranked by absolute correlation.
    """
    return series.reindex(
        series.abs().sort_values(ascending=False).index
    ).head(n)


def print_top_features(
    factor_id,
    Z,
    df_rna,
    df_adt,
    x_bc_3d,
    timepoints,
    lineages,
    n_rna=10,
    n_adt=20,
    n_fate=20,
):
    """
    Print top correlated features for a given factor.

    Parameters
    ----------
    factor_id : int
        Index of latent factor.
    Z : np.ndarray
        Factor matrix of shape (cells, factors).
    df_rna : pd.DataFrame
        RNA features.
    df_adt : pd.DataFrame
        ADT features.
    x_bc_3d : np.ndarray
        Fate tensor.
    timepoints : list
    lineages : list
    n_rna : int
    n_adt : int
    n_fate : int
    """

    factor_name = f"Factor {factor_id}"

    # reshape fate matrix
    X_fate = x_bc_3d.reshape(x_bc_3d.shape[0], -1)

    fate_features = [
        f"{t}_{l}"
        for t in timepoints
        for l in lineages
    ]

    # selected factor
    z = Z[:, factor_id]

    # correlations
    rna_corr = corr_with_view(z, df_rna.values, df_rna.columns)
    adt_corr = corr_with_view(z, df_adt.values, df_adt.columns)
    fate_corr = corr_with_view(z, X_fate, fate_features)

    # top features
    top_rna = top_n(rna_corr, n_rna)
    top_adt = top_n(adt_corr, n_adt)
    top_fate = top_n(fate_corr, n_fate)

    # print results
    print(f"\n=== Top RNA features for {factor_name} ===")
    print(top_rna)

    print(f"\n=== Top ADT features for {factor_name} ===")
    print(top_adt)

    print(f"\n=== Top Fate features for {factor_name} ===")
    print(top_fate)

    return {
        "RNA": top_rna,
        "ADT": top_adt,
        "Fate": top_fate,
    }

In [ ]:
# for each factor: 
results = print_top_features(
    factor_id=13,
    Z=Z,
    df_rna=df_rna,
    df_adt=df_adt,
    x_bc_3d=x_bc_3d,
    timepoints=timepoints,
    lineages=lineages,
    n_rna=10,
    n_adt=20,
    n_fate=20,
)